# Before running
If you haven't yet downloaded the COCO dataset, run `download_COCO.sh` in the top-level of the repo.

I just chose to use the validation set because its 1GB as opposed to the 18GB training dataset

This is what i did in terminal inside my nanochat uv environment before running this notebook:
```
uv pip install pycocotools
uv pip install transformers
uv pip install "numpy<2.0.0"
```

In [11]:
import sys
import os

# Get the absolute path to the outer 'nanochat' repo folder
nanochat_repo_path = os.path.abspath("nanochat")

# Add it to Python's path so it can find the inner 'nanochat' package
if nanochat_repo_path not in sys.path:
    sys.path.append(nanochat_repo_path)

# Now Python can see 'nanochat' as a top-level package!
from nanochat.tokenizer import get_tokenizer

In [12]:
import torch
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
from PIL import Image
import os
from transformers import CLIPProcessor, CLIPVisionModel

In [13]:
# Define a placeholder tokenizer for demonstration
# In practice, use a tokenizer compatible with nanochat
class PlaceholderTokenizer:
    def __init__(self):
        self.vocab = {"<SOS>": 1, "<EOS>": 2, "<PAD>": 0}  # Example vocabulary
        self.id_counter = 3

    def encode(self, text):
        tokens = [self.vocab.get(t, 0) for t in text.lower().split()] 
        return [self.vocab['<SOS>']] + tokens + [self.vocab['<EOS>']]

In [14]:
class CocoClipDataset(Dataset):
    def __init__(self, images_dir, ann_file, clip_model_name="openai/clip-vit-base-patch32"):
        self.coco = COCO(ann_file)
        self.images_dir = images_dir
        self.ids = list(self.coco.imgs.keys())
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.vision_model = CLIPVisionModel.from_pretrained(clip_model_name).to(self.device)
        self.processor = CLIPProcessor.from_pretrained(clip_model_name)
        
        # self.tokenizer = PlaceholderTokenizer()  # Replace with actual tokenizer for nanochat
        self.tokenizer = get_tokenizer() # use nanochat's tokenizer
        
        self.vision_model
        

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]

        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.images_dir, img_info['file_name'])
        image = Image.open(img_path).convert("RGB")
        
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)
        
        with torch.no_grad():
            vision_outputs = self.vision_model(**inputs)
            image_embeddings = vision_outputs.last_hidden_state
            
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        # Each image has 5 captions. Just take first 
        caption = anns[0]['caption']
        
        # for PlaceholderTokenizer
        # caption_tokens = self.tokenizer.encode(caption) 
        
        # for nanochat's tokenizer
        # --- NEW CODE ---
        # Encode using nanochat's tokenization logic
        caption_tokens = self.tokenizer.encode(
            caption, 
            prepend="<|bos|>", 
            append="<|assistant_end|>"
        )
        
        
        return {"image_embeddings": image_embeddings.squeeze(0), "caption_tokens": caption_tokens}

In [15]:
IMAGE_DIR = "COCO_data/val2017"
ANN_FILE = "COCO_data/annotations/captions_val2017.json"

# Just load one sample for testing

In [16]:

if not os.path.exists(IMAGE_DIR) or not os.path.exists(ANN_FILE):
    print("Please update IMAGE_DIR and ANN_FILE with the correct paths to your COCO dataset.")
    
else:
    coco_pipeline = CocoClipDataset(IMAGE_DIR, ANN_FILE)
    sample = coco_pipeline[0]
    
    # print the shapes and types to verify
    print("Image Embeddings Shape:", sample["image_embeddings"].shape)
    print("Caption Tokens:", sample["caption_tokens"])
    
    print(f"type of image_embeddings: {type(sample['image_embeddings'])}")
    print(f"shape of image_embeddings: {sample['image_embeddings'].shape}")

    # Expected shape: [num_patches + 1, embedding_dim] (e.g., [50, 768]) for CLIP ViT-B/32
    # The +1 is for the CLS token
    
    print(f"type of caption_tokens: {type(sample['caption_tokens'])}")
    print(f"shape of caption_tokens: {sample['caption_tokens'].shape if isinstance(sample['caption_tokens'], torch.Tensor) else 'N/A'}")
    # Expected type: list of token IDs (e.g., [1, 5, 101, 2]) where 1 is <SOS>, 2 is <EOS>, and others are word tokens
    
    print(f"content of caption_tokens: {sample['caption_tokens']}")
    

2026-04-28 23:24:16,150 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-28 23:24:16,168 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-base-patch32/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/config.json "HTTP/1.1 200 OK"


loading annotations into memory...
Done (t=0.06s)
creating index...
index created!


2026-04-28 23:24:16,219 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-04-28 23:24:16,271 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-04-28 23:24:16,313 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"
2026-04-28 23:24:16,363 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-04-28 23:24:16,418 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-base-patch32 "HTTP/1.1 200 OK"
2026-04-28 23:24:16,478 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/commits/main "HTTP/1.1 200 OK"
2026-04-28 23:24:16,549 - ht

Image Embeddings Shape: torch.Size([50, 768])
Caption Tokens: [65527, 65, 529, 309, 283, 257, 9190, 1645, 55376, 294, 46, 65531]
type of image_embeddings: <class 'torch.Tensor'>
shape of image_embeddings: torch.Size([50, 768])
type of caption_tokens: <class 'list'>
shape of caption_tokens: N/A
content of caption_tokens: [65527, 65, 529, 309, 283, 257, 9190, 1645, 55376, 294, 46, 65531]


# Load into Pytorch DataLoader

In [17]:
# collate function to handle batches with variable-length captions
from torch.nn.utils.rnn import pad_sequence

def collate_batch(batch):
    # Grab all the image embeddings and stack them (since they are identical shapes)
    image_embeddings = [item["image_embeddings"] for item in batch]
    batched_images = torch.stack(image_embeddings)
    
    # Grab all the captions and convert them from Python lists to PyTorch Tensors
    caption_tokens = [torch.tensor(item["caption_tokens"]) for item in batch]
    
    # Pad the shorter captions with 0s so they all match the longest caption in the batch
    batched_captions = pad_sequence(caption_tokens, batch_first=True, padding_value=0)
    
    return {
        "image_embeddings": batched_images,
        "caption_tokens": batched_captions
    }

In [18]:
data_loader = DataLoader(coco_pipeline, batch_size=4, shuffle=True, collate_fn=collate_batch)

batch = next(iter(data_loader))
print("Batch Image Embeddings Shape:", batch["image_embeddings"].shape)
print("Batch Caption Tokens Shape:", batch["caption_tokens"].shape)

Batch Image Embeddings Shape: torch.Size([4, 50, 768])
Batch Caption Tokens Shape: torch.Size([4, 14])


In [21]:
# Save CLIP image embeddings and all caption tokens (one file per caption) for offline SFT
import os
save_dir = 'COCO_data/embeddings'
os.makedirs(save_dir, exist_ok=True)
print('Saving embeddings and captions to', save_dir)
n = len(coco_pipeline)
for i, img_id in enumerate(coco_pipeline.ids):
    # Save image embedding once (compute if missing)
    save_path = os.path.join(save_dir, f'{img_id}.pt')
    if not os.path.exists(save_path):
        img_info = coco_pipeline.coco.loadImgs(img_id)[0]
        img_path = os.path.join(IMAGE_DIR, img_info['file_name'])
        image = Image.open(img_path).convert('RGB')
        inputs = coco_pipeline.processor(images=image, return_tensors='pt').to(coco_pipeline.device)
        with torch.no_grad():
            vision_outputs = coco_pipeline.vision_model(**inputs)
            image_embeddings = vision_outputs.last_hidden_state.squeeze(0).cpu()
        torch.save(image_embeddings, save_path)
    else:
        # If embedding exists, load it to verify shape (optional)
        image_embeddings = torch.load(save_path)

    # Save each caption for this image as a separate _cap_<idx>.pt file
    ann_ids = coco_pipeline.coco.getAnnIds(imgIds=img_id)
    anns = coco_pipeline.coco.loadAnns(ann_ids)
    for j, ann in enumerate(anns):
        cap_path = os.path.join(save_dir, f'{img_id}_cap_{j}.pt')
        if os.path.exists(cap_path):
            continue
        caption = ann.get('caption', '').strip()
        if not caption:
            continue
        caption_tokens = coco_pipeline.tokenizer.encode(caption, prepend='<|bos|>', append='<|assistant_end|>')
        caption_tokens_tensor = torch.tensor(caption_tokens, dtype=torch.long)
        torch.save(caption_tokens_tensor, cap_path)

    if (i+1) % 100 == 0:
        print(f'Saved {i+1}/{n} items')
print('Done.')

Saving embeddings and captions to COCO_data/embeddings
Saved 100/5000 items
Saved 200/5000 items
Saved 300/5000 items
Saved 400/5000 items
Saved 500/5000 items
Saved 600/5000 items
Saved 700/5000 items
Saved 800/5000 items
Saved 900/5000 items
Saved 1000/5000 items
Saved 1100/5000 items
Saved 1200/5000 items
Saved 1300/5000 items
Saved 1400/5000 items
Saved 1500/5000 items
Saved 1600/5000 items
Saved 1700/5000 items
Saved 1800/5000 items
Saved 1900/5000 items
Saved 2000/5000 items
Saved 2100/5000 items
Saved 2200/5000 items
Saved 2300/5000 items
Saved 2400/5000 items
Saved 2500/5000 items
Saved 2600/5000 items
Saved 2700/5000 items
Saved 2800/5000 items
Saved 2900/5000 items
Saved 3000/5000 items
Saved 3100/5000 items
Saved 3200/5000 items
Saved 3300/5000 items
Saved 3400/5000 items
Saved 3500/5000 items
Saved 3600/5000 items
Saved 3700/5000 items
Saved 3800/5000 items
Saved 3900/5000 items
Saved 4000/5000 items
Saved 4100/5000 items
Saved 4200/5000 items
Saved 4300/5000 items
Saved 44